# Capacidad 3 — Entrenamiento de modelos de predicción (SECOP CTeI)

**Objetivo del reto (Cap. 3):** predicciones sobre tipo de contratación, rangos de presupuesto,
probabilidad de adjudicación y sectores con mayor probabilidad de inversión.

Este notebook **entrena y compara modelos** paso a paso y deja una bitácora de hallazgos.

## Modelos

| # | Target | Tipo | Modelos |
|---|--------|------|---------|
| A | `adjudicado_proceso` | Binaria | Trivial → Regla modalidad → Logística → HistGradientBoosting (+ LightGBM si está) |
| B | Rango de presupuesto | Multiclase (bins) | Trivial / HGB |
| C | Segmento UNSPSC | Multiclase **sin** usar segmento como feature | Trivial / HGB |

## Reglas

1. Universo **competitivo**
2. Split **temporal**
3. Features solo al **publicar** (sin fuga)
4. Preferir `secop_ctei_procesos_deflactado_sin_implausibles.csv`
5. Target encoding de entidad **solo con train**

Kernel: **Python 3.12** + `scikit-learn`, `pandas`.

In [1]:
from __future__ import annotations

from pathlib import Path
import warnings
import time

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
FECHA_CORTE = pd.Timestamp("2025-07-01")
OUT_DIR = Path("salidas_capacidad3")
OUT_DIR.mkdir(exist_ok=True)

HALLAZGOS: list[dict] = []

def registrar(paso: str, hallazgo: str, **metrics):
    fila = {"paso": paso, "hallazgo": hallazgo, **metrics}
    HALLAZGOS.append(fila)
    print(f"[{paso}] {hallazgo}")
    if metrics:
        nice = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in metrics.items()}
        print("   ", nice)

try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False
    lgb = None

print(f"LightGBM disponible: {HAS_LGBM}")
print(f"Salidas -> {OUT_DIR.resolve()}")

LightGBM disponible: True
Salidas -> /content/salidas_capacidad3


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1. Carga del dataset monetario limpio

Prioridad: `*_deflactado_sin_implausibles.csv`. Si no existe, se usa el deflactado
completo y se aplica el filtro de plausibilidad en memoria.

In [3]:
CANDIDATOS = [
    Path("/content/drive/MyDrive/archivo_descomprimido/secop_ctei_procesos_deflactado_sin_implausibles.csv"),
    Path("/content/drive/MyDrive/archivo_descomprimido/secop_ctei_procesos_deflactado.csv"),
]
ruta = next((p for p in CANDIDATOS if p.exists()), None)
if ruta is None:
    raise FileNotFoundError("No hay CSV deflactado en esta carpeta.")

df_raw = pd.read_csv(ruta, low_memory=False)
df_raw["fecha_de_publicacion_del"] = pd.to_datetime(
    df_raw["fecha_de_publicacion_del"], errors="coerce"
)

print(f"Archivo: {ruta.name}")
print(f"Filas: {len(df_raw):,} | Columnas: {df_raw.shape[1]}")

if "sin_implausibles" not in ruta.name:
    UMBRAL_ABSOLUTO = 1e13
    RATIO_MAX = 100
    valor = pd.to_numeric(df_raw.get("valor_adjudicado_total"), errors="coerce")
    precio = pd.to_numeric(df_raw.get("precio_base"), errors="coerce")
    flag = valor.gt(UMBRAL_ABSOLUTO) | (precio.gt(1e6) & valor.gt(RATIO_MAX * precio))
    n_out = int(flag.fillna(False).sum())
    df_raw = df_raw.loc[~flag.fillna(False)].copy()
    registrar(
        "1.carga",
        f"Filtro implausibles en memoria: excluidos {n_out} procesos.",
        n_excluidos=n_out,
        n_restantes=len(df_raw),
    )
else:
    registrar("1.carga", f"CSV sin implausibles ({ruta.name}).", n=len(df_raw))

Archivo: secop_ctei_procesos_deflactado_sin_implausibles.csv
Filas: 492,784 | Columnas: 37
[1.carga] CSV sin implausibles (secop_ctei_procesos_deflactado_sin_implausibles.csv).
    {'n': 492784}


# 2. Universo competitivo

Fuera de estas modalidades, `adjudicado` es estructuralmente vacío (EDA) — no hay señal.

In [4]:
MODALIDADES_COMPETITIVAS = [
    "Licitación pública", "Licitación pública Obra Publica",
    "Licitación Pública Acuerdo Marco de Precios",
    "Concurso de méritos abierto", "Concurso de méritos con precalificación",
    "Selección Abreviada de Menor Cuantía",
    "Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes",
    "Selección abreviada subasta inversa", "Mínima cuantía",
    "Contratación Directa (con ofertas)", "Contratación régimen especial (con ofertas)",
]

df = df_raw[df_raw["modalidad_de_contratacion"].isin(MODALIDADES_COMPETITIVAS)].copy()
df = df[df["fecha_de_publicacion_del"].notna()].copy()
df["y_adj"] = df["adjudicado_proceso"].astype(str).isin(["True", "true", "1", "1.0"])

print(f"Competitivos con fecha: {len(df):,} ({len(df)/len(df_raw):.1%} del CSV)")
print(f"Tasa adjudicación bruta: {df['y_adj'].mean():.1%}")
print("\nTop modalidades:")
print(
    df.groupby("modalidad_de_contratacion")["y_adj"]
      .agg(tasa="mean", n="size")
      .assign(tasa_pct=lambda d: (d["tasa"] * 100).round(1))
      .sort_values("n", ascending=False)
      .head(12)
)

registrar(
    "2.universo",
    "Universo competitivo listo; la tasa bruta incluye procesos abiertos (censura a la derecha).",
    n=len(df),
    tasa_adjudicacion=float(df["y_adj"].mean()),
)

Competitivos con fecha: 69,671 (14.1% del CSV)
Tasa adjudicación bruta: 55.3%

Top modalidades:
                                                     tasa      n  tasa_pct
modalidad_de_contratacion                                                 
Mínima cuantía                                     0.7952  21874   79.5000
Concurso de méritos abierto                        0.4061  16372   40.6000
Selección Abreviada de Menor Cuantía               0.1001  10211   10.0000
Contratación Directa (con ofertas)                 0.8327   7706   83.3000
Contratación régimen especial (con ofertas)        0.6879   5395   68.8000
Selección abreviada subasta inversa                0.4138   4729   41.4000
Licitación pública                                 0.4121   2048   41.2000
Licitación pública Obra Publica                    0.4094   1143   40.9000
Seleccion Abreviada Menor Cuantia Sin Manifesta... 0.3750    152   37.5000
Concurso de méritos con precalificación            0.2353     34   23.5000
Lici

# 3. Variante: solo procesos resueltos

Muchos `False` son procesos aún abiertos. Comparamos con un subconjunto terminal
(Adjudicado / Cancelado) para ver el efecto sobre métricas.

In [5]:
estado_res = df.get("estado_resumen", pd.Series(index=df.index, dtype=object)).astype(str)
estado_proc = df.get("estado_del_procedimiento", pd.Series(index=df.index, dtype=object)).astype(str)
mask_resuelto = estado_res.str.contains("Adjudicado", case=False, na=False) | estado_proc.eq("Cancelado")
df_resuelto = df.loc[mask_resuelto].copy()

print(f"Resueltos: {len(df_resuelto):,} | tasa: {df_resuelto['y_adj'].mean():.1%}")
print(f"No resueltos: {(~mask_resuelto).sum():,}")

registrar(
    "3.censura",
    "Subconjunto resuelto concentra señal; el completo tiene más 'No' por apertura.",
    n_resuelto=len(df_resuelto),
    tasa_resuelto=float(df_resuelto["y_adj"].mean()) if len(df_resuelto) else None,
    n_abiertos=int((~mask_resuelto).sum()),
)

Resueltos: 43,566 | tasa: 86.3%
No resueltos: 26,105
[3.censura] Subconjunto resuelto concentra señal; el completo tiene más 'No' por apertura.
    {'n_resuelto': 43566, 'tasa_resuelto': 0.863, 'n_abiertos': 26105}


# 4. Features sin fuga + helpers

**Prohibido:** valor adjudicado, fecha adjudicación, nº proveedores adjudicados,
respuestas al procedimiento, señales post-cierre.

**Incluido:** log precio real, duración, lotes, modalidad, depto agrupado, mes/año,
target encoding de entidad (fit solo en train).

In [6]:
FEATURES_NUM = [
    "log_precio_base_real", "duracion", "numero_de_lotes",
    "mes_publicacion", "anio_publicacion",
]
FEATURES_CAT = ["modalidad_de_contratacion", "departamento_agrupado"]


def preparar_base(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["precio_base_real"] = pd.to_numeric(out["precio_base_real"], errors="coerce")
    out = out[out["precio_base_real"].fillna(-1) > 0].copy()
    out["log_precio_base_real"] = np.log1p(out["precio_base_real"])
    out["duracion"] = pd.to_numeric(out["duracion"], errors="coerce")
    out["numero_de_lotes"] = pd.to_numeric(out["numero_de_lotes"], errors="coerce")
    out["mes_publicacion"] = out["fecha_de_publicacion_del"].dt.month
    out["anio_publicacion"] = out["fecha_de_publicacion_del"].dt.year
    top_dep = out["departamento_entidad"].value_counts().head(20).index
    out["departamento_agrupado"] = out["departamento_entidad"].where(
        out["departamento_entidad"].isin(top_dep), "OTRO"
    )
    out["entidad"] = out["entidad"].fillna("SIN_ENTIDAD").astype(str)
    out["segmento_unspsc"] = out["segmento_unspsc"].astype(str)
    return out


df = preparar_base(df)
df_resuelto = preparar_base(df_resuelto)
print(df[FEATURES_NUM].describe().T[["count", "mean", "50%", "max"]])

                           count       mean        50%             max
log_precio_base_real 69,375.0000    18.9877    18.9429         29.8649
duracion             69,375.0000 1,277.3778     6.0000 86,111,500.0000
numero_de_lotes      69,375.0000     0.1082     0.0000        113.0000
mes_publicacion      69,375.0000     7.1500     7.0000         12.0000
anio_publicacion     69,375.0000 2,023.7616 2,024.0000      2,026.0000


In [7]:
def split_temporal(frame: pd.DataFrame, fecha_corte=FECHA_CORTE):
    train = frame[frame["fecha_de_publicacion_del"] < fecha_corte].copy()
    test = frame[frame["fecha_de_publicacion_del"] >= fecha_corte].copy()
    return train, test


def imputar_con_train(train, test, cols=("duracion", "numero_de_lotes")):
    train, test = train.copy(), test.copy()
    for col in cols:
        med = train[col].median()
        train[col] = train[col].fillna(med)
        test[col] = test[col].fillna(med)
    return train, test


def target_encode_entidad(train, test, y_col="y_adj", alpha=20.0):
    train, test = train.copy(), test.copy()
    global_mean = train[y_col].mean()
    stats = train.groupby("entidad")[y_col].agg(["sum", "count"])
    stats["te"] = (stats["sum"] + alpha * global_mean) / (stats["count"] + alpha)
    mapping = stats["te"].to_dict()
    train["entidad_te"] = train["entidad"].map(mapping).fillna(global_mean)
    test["entidad_te"] = test["entidad"].map(mapping).fillna(global_mean)
    return train, test


def matrices_X(train, test, extra_num=None):
    extra_num = extra_num or []
    num_cols = FEATURES_NUM + extra_num
    cat_train = pd.get_dummies(train[FEATURES_CAT], drop_first=True)
    cat_test = pd.get_dummies(test[FEATURES_CAT], drop_first=True)
    cat_test = cat_test.reindex(columns=cat_train.columns, fill_value=0)
    X_train = pd.concat(
        [train[num_cols].reset_index(drop=True), cat_train.reset_index(drop=True)],
        axis=1,
    )
    X_test = pd.concat(
        [test[num_cols].reset_index(drop=True), cat_test.reset_index(drop=True)],
        axis=1,
    )
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    return X_train, X_test


def metricas_binarias(y_true, y_pred, y_prob, nombre: str) -> dict:
    out = {
        "modelo": nombre,
        "n_test": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }
    if len(np.unique(y_true)) > 1 and y_prob is not None:
        out["auc_roc"] = float(roc_auc_score(y_true, y_prob))
        out["auc_pr"] = float(average_precision_score(y_true, y_prob))
    else:
        out["auc_roc"] = np.nan
        out["auc_pr"] = np.nan
    return out

print("Helpers listos.")

Helpers listos.


# 5. Target A — Probabilidad de adjudicación

Comparamos baselines y boosting en universo completo y (si hay datos) solo resueltos.

In [8]:
def pipeline_adjudicacion(frame: pd.DataFrame, etiqueta_universo: str) -> pd.DataFrame:
    train, test = split_temporal(frame)
    train, test = imputar_con_train(train, test)
    train, test = target_encode_entidad(train, test)

    print(f"\n===== Universo: {etiqueta_universo} =====")
    print(
        f"Train: {len(train):,} | {train['fecha_de_publicacion_del'].min().date()} -> "
        f"{train['fecha_de_publicacion_del'].max().date()} | tasa={train['y_adj'].mean():.1%}"
    )
    print(
        f"Test:  {len(test):,} | {test['fecha_de_publicacion_del'].min().date()} -> "
        f"{test['fecha_de_publicacion_del'].max().date()} | tasa={test['y_adj'].mean():.1%}"
    )

    y_train = train["y_adj"].astype(int)
    y_test = test["y_adj"].astype(int)
    resultados = []

    clase_may = int(y_train.mode()[0])
    pred_triv = np.full(len(y_test), clase_may)
    resultados.append(metricas_binarias(y_test, pred_triv, None, "1.trivial"))

    tasa_mod = train.groupby("modalidad_de_contratacion")["y_adj"].mean()
    prob_reg = test["modalidad_de_contratacion"].map(tasa_mod).fillna(y_train.mean())
    pred_reg = (prob_reg >= 0.5).astype(int)
    resultados.append(metricas_binarias(y_test, pred_reg, prob_reg, "2.regla_modalidad"))

    X_train, X_test = matrices_X(train, test, extra_num=["entidad_te"])

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(X_train)
    Xte_s = scaler.transform(X_test)
    logit = LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    )
    t0 = time.time()
    logit.fit(Xtr_s, y_train)
    prob_logit = logit.predict_proba(Xte_s)[:, 1]
    pred_logit = (prob_logit >= 0.5).astype(int)
    m_logit = metricas_binarias(y_test, pred_logit, prob_logit, "3.logistica")
    m_logit["segundos"] = time.time() - t0
    resultados.append(m_logit)

    hgb = HistGradientBoostingClassifier(
        max_depth=6,
        learning_rate=0.08,
        max_iter=200,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    )
    t0 = time.time()
    hgb.fit(X_train, y_train)
    prob_hgb = hgb.predict_proba(X_test)[:, 1]
    pred_hgb = (prob_hgb >= 0.5).astype(int)
    m_hgb = metricas_binarias(y_test, pred_hgb, prob_hgb, "4.hist_gradient_boosting")
    m_hgb["segundos"] = time.time() - t0
    resultados.append(m_hgb)

    pred_best = pred_hgb
    if HAS_LGBM:
        clf = lgb.LGBMClassifier(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            verbose=-1,
        )
        t0 = time.time()
        clf.fit(X_train, y_train)
        prob_lgb = clf.predict_proba(X_test)[:, 1]
        pred_lgb = (prob_lgb >= 0.5).astype(int)
        m_lgb = metricas_binarias(y_test, pred_lgb, prob_lgb, "5.lightgbm")
        m_lgb["segundos"] = time.time() - t0
        resultados.append(m_lgb)
        pred_best = pred_lgb

    tabla = pd.DataFrame(resultados)
    tabla["universo"] = etiqueta_universo
    print("\nComparacion:")
    print(tabla[["modelo", "accuracy", "f1", "auc_roc", "auc_pr"]].to_string(index=False))

    score_col = "auc_pr" if tabla["auc_pr"].notna().any() else "auc_roc"
    best_name = tabla.sort_values(score_col, ascending=False).iloc[0]["modelo"]
    print(f"\nMejor por {score_col}: {best_name}")
    if best_name.startswith("5.") and HAS_LGBM:
        cm = confusion_matrix(y_test, pred_lgb)
    elif best_name.startswith("4."):
        cm = confusion_matrix(y_test, pred_hgb)
    elif best_name.startswith("3."):
        cm = confusion_matrix(y_test, pred_logit)
    elif best_name.startswith("2."):
        cm = confusion_matrix(y_test, pred_reg)
    else:
        cm = confusion_matrix(y_test, pred_triv)
    print(pd.DataFrame(cm, index=["Real No", "Real Si"], columns=["Pred No", "Pred Si"]))

    best_auc = tabla.sort_values("auc_roc", ascending=False).iloc[0]
    registrar(
        f"5.adj.{etiqueta_universo}",
        f"Mejor={best_auc['modelo']} | AUC={best_auc['auc_roc']:.3f} | Acc={best_auc['accuracy']:.1%}. "
        "Accuracy alta del trivial no implica poder predictivo bajo desbalance.",
        auc_roc=float(best_auc["auc_roc"]) if pd.notna(best_auc["auc_roc"]) else None,
        accuracy=float(best_auc["accuracy"]),
        n_train=len(train),
        n_test=len(test),
    )
    return tabla


tabla_full = pipeline_adjudicacion(df, "competitivo_completo")
tabla_res = (
    pipeline_adjudicacion(df_resuelto, "solo_resueltos")
    if len(df_resuelto) > 1000
    else None
)

tablas_adj = [tabla_full] + ([tabla_res] if tabla_res is not None else [])
resumen_adj = pd.concat(tablas_adj, ignore_index=True)
resumen_adj.to_csv(OUT_DIR / "metricas_adjudicacion.csv", index=False, encoding="utf-8-sig")
print(f"\nGuardado: {OUT_DIR / 'metricas_adjudicacion.csv'}")
resumen_adj


===== Universo: competitivo_completo =====
Train: 51,813 | 2022-01-03 -> 2025-06-30 | tasa=55.4%
Test:  17,562 | 2025-07-01 -> 2026-07-29 | tasa=55.2%

Comparacion:
                  modelo  accuracy     f1  auc_roc  auc_pr
               1.trivial    0.5517 0.7111      NaN     NaN
       2.regla_modalidad    0.7415 0.7627   0.7843  0.7618
             3.logistica    0.7511 0.7695   0.8056  0.8076
4.hist_gradient_boosting    0.7520 0.7687   0.8106  0.8102
              5.lightgbm    0.7537 0.7718   0.8111  0.8081

Mejor por auc_pr: 4.hist_gradient_boosting
         Pred No  Pred Si
Real No     5969     1904
Real Si     2451     7238
[5.adj.competitivo_completo] Mejor=5.lightgbm | AUC=0.811 | Acc=75.4%. Accuracy alta del trivial no implica poder predictivo bajo desbalance.
    {'auc_roc': 0.8111, 'accuracy': 0.7537, 'n_train': 51813, 'n_test': 17562}

===== Universo: solo_resueltos =====
Train: 32,709 | 2022-01-03 -> 2025-06-30 | tasa=86.0%
Test:  10,706 | 2025-07-01 -> 2026-07-28 | ta

,modelo,n_test,accuracy,f1,auc_roc,auc_pr,segundos,universo
0,1.trivial,17562,0.5517,0.7111,NaN,NaN,NaN,competitivo_completo
1,2.regla_modalidad,17562,0.7415,0.7627,0.7843,0.7618,NaN,competitivo_completo
2,3.logistica,17562,0.7511,0.7695,0.8056,0.8076,0.2868,competitivo_completo
3,4.hist_gradient_boosting,17562,0.7520,0.7687,0.8106,0.8102,2.0823,competitivo_completo
4,5.lightgbm,17562,0.7537,0.7718,0.8111,0.8081,3.1297,competitivo_completo
5,1.trivial,10706,0.8726,0.9320,NaN,NaN,NaN,solo_resueltos
6,2.regla_modalidad,10706,0.8726,0.9320,0.5551,0.8881,NaN,solo_resueltos
7,3.logistica,10706,0.7070,0.8181,0.6092,0.9078,0.2209,solo_resueltos
8,4.hist_gradient_boosting,10706,0.6735,0.7912,0.6041,0.9021,1.3670,solo_resueltos
9,5.lightgbm,10706,0.7090,0.8198,0.6065,0.9035,1.3141,solo_resueltos


## 5.1 Cómo leer adjudicación

- **AUC ~0.50–0.60** → separación débil; SECOP tabular satura pronto.
- Si HGB/LightGBM ≈ regla por modalidad → la modalidad ya explica casi todo.
- `solo_resueltos` suele subir accuracy pero cambia el significado del target.

# 6. Target B — Rangos de presupuesto

Bins por cuantiles de **train**. No se usa el monto como feature (trivializaría el target).

In [9]:
def pipeline_presupuesto(frame: pd.DataFrame) -> pd.DataFrame:
    train, test = split_temporal(frame)
    train, test = imputar_con_train(train, test)
    train, test = target_encode_entidad(train, test, y_col="y_adj")

    qs = train["precio_base_real"].quantile([0.25, 0.5, 0.75]).tolist()
    bins = [-np.inf, qs[0], qs[1], qs[2], np.inf]
    labels = ["Q1_bajo", "Q2", "Q3", "Q4_alto"]
    train = train.copy()
    test = test.copy()
    train["y_presupuesto"] = pd.cut(train["precio_base_real"], bins=bins, labels=labels)
    test["y_presupuesto"] = pd.cut(test["precio_base_real"], bins=bins, labels=labels)
    train = train[train["y_presupuesto"].notna()]
    test = test[test["y_presupuesto"].notna()]

    print("Cortes train (COP reales):", [f"{x:,.0f}" for x in qs])
    print("Distribucion test:")
    print(test["y_presupuesto"].value_counts(normalize=True).sort_index().mul(100).round(1))

    num = ["duracion", "numero_de_lotes", "mes_publicacion", "anio_publicacion", "entidad_te"]
    cat_train = pd.get_dummies(train[FEATURES_CAT], drop_first=True)
    cat_test = pd.get_dummies(test[FEATURES_CAT], drop_first=True).reindex(
        columns=cat_train.columns, fill_value=0
    )
    X_train = pd.concat([train[num].reset_index(drop=True), cat_train.reset_index(drop=True)], axis=1)
    X_test = pd.concat([test[num].reset_index(drop=True), cat_test.reset_index(drop=True)], axis=1)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    y_train = train["y_presupuesto"].astype(str)
    y_test = test["y_presupuesto"].astype(str)

    maj = y_train.mode()[0]
    acc_triv = accuracy_score(y_test, np.full(len(y_test), maj))

    hgb = HistGradientBoostingClassifier(
        max_depth=6, learning_rate=0.08, max_iter=200, random_state=RANDOM_STATE
    )
    hgb.fit(X_train, y_train)
    pred = hgb.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1m = f1_score(y_test, pred, average="macro", zero_division=0)

    print(f"\nTrivial: {acc_triv:.1%} | HGB: {acc:.1%} | F1-macro: {f1m:.3f}")
    print(classification_report(y_test, pred, zero_division=0))

    registrar(
        "6.presupuesto",
        "Bins Q1-Q4 sin usar monto como feature; proxy de rango presupuestal condicionado al contexto.",
        acc_trivial=float(acc_triv),
        acc_hgb=float(acc),
        f1_macro=float(f1m),
    )
    return pd.DataFrame([
        {"modelo": "trivial", "accuracy": acc_triv, "f1_macro": np.nan},
        {"modelo": "hist_gradient_boosting", "accuracy": acc, "f1_macro": f1m},
    ])


tabla_pres = pipeline_presupuesto(df)
tabla_pres.to_csv(OUT_DIR / "metricas_presupuesto_bins.csv", index=False, encoding="utf-8-sig")
tabla_pres

Cortes train (COP reales): ['46,899,281', '174,868,365', '588,023,449']
Distribucion test:
y_presupuesto
Q1_bajo   27.7000
Q2        24.8000
Q3        23.6000
Q4_alto   23.8000
Name: proportion, dtype: float64

Trivial: 27.7% | HGB: 61.3% | F1-macro: 0.591
              precision    recall  f1-score   support

     Q1_bajo       0.72      0.84      0.78      4873
          Q2       0.54      0.28      0.37      4363
          Q3       0.48      0.63      0.55      4150
     Q4_alto       0.66      0.68      0.67      4176

    accuracy                           0.61     17562
   macro avg       0.60      0.61      0.59     17562
weighted avg       0.61      0.61      0.60     17562

[6.presupuesto] Bins Q1-Q4 sin usar monto como feature; proxy de rango presupuestal condicionado al contexto.
    {'acc_trivial': 0.2775, 'acc_hgb': 0.6126, 'f1_macro': 0.5911}


,modelo,accuracy,f1_macro
0,trivial,0.2775,NaN
1,hist_gradient_boosting,0.6126,0.5911


# 7. Target C — Segmento UNSPSC sin usar el segmento como input

Mide si modalidad/monto/calendario/entidad predicen el sector. Si apenas supera al trivial,
la taxonomía semántica por embeddings aporta más que seguir empujando tabular.

In [10]:
def pipeline_segmento(frame: pd.DataFrame) -> pd.DataFrame:
    train, test = split_temporal(frame)
    train, test = imputar_con_train(train, test)
    train, test = target_encode_entidad(train, test)

    train = train[train["segmento_unspsc"].isin(["80", "81", "86"])].copy()
    test = test[test["segmento_unspsc"].isin(["80", "81", "86"])].copy()

    print("Distribucion segmento test:")
    print(test["segmento_unspsc"].value_counts(normalize=True).mul(100).round(1))

    num = [
        "log_precio_base_real", "duracion", "numero_de_lotes",
        "mes_publicacion", "anio_publicacion", "entidad_te",
    ]
    cat_cols = ["modalidad_de_contratacion", "departamento_agrupado"]
    cat_train = pd.get_dummies(train[cat_cols], drop_first=True)
    cat_test = pd.get_dummies(test[cat_cols], drop_first=True).reindex(
        columns=cat_train.columns, fill_value=0
    )
    X_train = pd.concat([train[num].reset_index(drop=True), cat_train.reset_index(drop=True)], axis=1)
    X_test = pd.concat([test[num].reset_index(drop=True), cat_test.reset_index(drop=True)], axis=1)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    y_train = train["segmento_unspsc"].astype(str)
    y_test = test["segmento_unspsc"].astype(str)

    maj = y_train.mode()[0]
    acc_triv = accuracy_score(y_test, np.full(len(y_test), maj))

    hgb = HistGradientBoostingClassifier(
        max_depth=6, learning_rate=0.08, max_iter=250, random_state=RANDOM_STATE
    )
    hgb.fit(X_train, y_train)
    pred = hgb.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1m = f1_score(y_test, pred, average="macro", zero_division=0)

    print(f"\nTrivial (siempre {maj}): {acc_triv:.1%}")
    print(f"HGB: accuracy={acc:.1%} | F1-macro={f1m:.3f}")
    print(classification_report(y_test, pred, zero_division=0))

    registrar(
        "7.segmento",
        "Si HGB apenas supera al trivial, embeddings del objeto contractual son el siguiente frente.",
        acc_trivial=float(acc_triv),
        acc_hgb=float(acc),
        f1_macro=float(f1m),
    )
    return pd.DataFrame([
        {"modelo": "trivial", "accuracy": acc_triv, "f1_macro": np.nan},
        {"modelo": "hist_gradient_boosting", "accuracy": acc, "f1_macro": f1m},
    ])


tabla_seg = pipeline_segmento(df)
tabla_seg.to_csv(OUT_DIR / "metricas_segmento.csv", index=False, encoding="utf-8-sig")
tabla_seg

Distribucion segmento test:
segmento_unspsc
81   62.5000
80   26.8000
86   10.7000
Name: proportion, dtype: float64

Trivial (siempre 81): 62.5%
HGB: accuracy=68.4% | F1-macro=0.518
              precision    recall  f1-score   support

          80       0.61      0.37      0.46      4710
          81       0.71      0.90      0.79     10970
          86       0.55      0.21      0.30      1882

    accuracy                           0.68     17562
   macro avg       0.62      0.49      0.52     17562
weighted avg       0.66      0.68      0.65     17562

[7.segmento] Si HGB apenas supera al trivial, embeddings del objeto contractual son el siguiente frente.
    {'acc_trivial': 0.6246, 'acc_hgb': 0.6837, 'f1_macro': 0.5176}


,modelo,accuracy,f1_macro
0,trivial,0.6246,NaN
1,hist_gradient_boosting,0.6837,0.5176


# 8. Bitácora de hallazgos + artefactos

In [11]:
bitacora = pd.DataFrame(HALLAZGOS)
bitacora.to_csv(OUT_DIR / "bitacora_hallazgos.csv", index=False, encoding="utf-8-sig")
bitacora.to_json(
    OUT_DIR / "bitacora_hallazgos.json", orient="records", force_ascii=False, indent=2
)
print(bitacora[["paso", "hallazgo"]].to_string(index=False))
print(f"\nArtefactos en {OUT_DIR.resolve()}:")
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)

                      paso                                                                                                          hallazgo
                   1.carga                                       CSV sin implausibles (secop_ctei_procesos_deflactado_sin_implausibles.csv).
                2.universo                       Universo competitivo listo; la tasa bruta incluye procesos abiertos (censura a la derecha).
                 3.censura                                    Subconjunto resuelto concentra señal; el completo tiene más 'No' por apertura.
5.adj.competitivo_completo  Mejor=5.lightgbm | AUC=0.811 | Acc=75.4%. Accuracy alta del trivial no implica poder predictivo bajo desbalance.
      5.adj.solo_resueltos Mejor=3.logistica | AUC=0.609 | Acc=70.7%. Accuracy alta del trivial no implica poder predictivo bajo desbalance.
             6.presupuesto                     Bins Q1-Q4 sin usar monto como feature; proxy de rango presupuestal condicionado al contexto.
             

## Conclusiones (completar tras ejecutar)

1. **Adjudicación:** mejor modelo=`___` AUC=`___` universo=`___`. ¿Supera a la regla por modalidad? Sí/No.
2. **Presupuesto:** HGB=`___` vs trivial=`___`.
3. **Segmento:** HGB=`___` vs trivial=`___`. Si la brecha es chica → embeddings (Cap. 2 semántica).
4. **Siguiente paso:** historial entidad×modalidad / más años SECOP / embeddings objeto / calibración.